# HeatUp 01 - structure preparation and MD

Prepares the inputs of the stability gates. Every step is cached (`manifest.json` = config fingerprint +
input hashes), so re-running a cell only recomputes what changed (or what `cfg.run.force` lists). Each step runs in
its own process, so GPU memory is released between phases.

| step | output (per `<material>/<label>/`) | used by | what it does |
|---|---|---|---|
| `relax` | `relaxation/CONTCAR`, `energy.json` | everything | BFGS (+ FrechetCellFilter) geometry optimisation |
| `elastic` | `elastic/elastic.json` (relaxed-ion) | Gate 1 | 6x6 Voigt stiffness tensor from strained cells |
| `phonons` | `phonons/ha.json` | Gates 2, 3 | harmonic modes (phonopy finite displacements); **always** computed, also gives the competing-phase free energies |
| `qha` (optional) | `phonons/qha.json` | Gate 3 with `hull.phonon_method="QHA"` | phonons at several volumes, F(V,T) fit |
| `fc3` (optional) | `phonons/fc3.json` | diagnostic (`report["anharmonic"]`) | third-order force constants (phono3py): kappa_RTA(T), phonon linewidths gamma/nu |
| `md` | `md/<T>K/output.traj` (positions, momenta, energy, stress) | Gates 0, 2 | NPT or NVT trajectory in a supercell |
| `analysis` (post-processing) | `md/<T>K/analysis.json`, `vdos.json` | Gates 0, 2, VDOS overlay | block averages, thermo series, VACF -> VDOS |

Every option of every step is listed in the **Configuration** cell below (defaults shown); change only what you need.

## Where the data lives
```
<database_root>/                    targets: read + write
└── <material>/<label>/             e.g. BaTiO3/P4mm
    ├── POSCAR                      the input structure
    ├── relaxation/   CONTCAR, energy.json, run.traj          step "relax"
    ├── elastic/      elastic.json                            step "elastic"   -> Gate 1
    ├── phonons/      ha.json, [qha.json], phonon.png         steps "phonons", "qha"
    ├── md/<T>K/      output.traj, ...                        step "md"        -> Gates 0, 2
    └── stability/    report_<stage>.json, result_<stage>.json, report_<stage>.png, zero_K_hull.json
<candidates_root>/                  user inputs: read-only, never written
<competing_root>/                   competing phases (MP downloads, generated, mirrored): same <material>/<label>/ layout,
                                    each with its own relaxation/ and phonons/ (they feed Gate 3)
```
Every step directory carries a `manifest.json` (config fingerprint + input hashes): a step is recomputed when its
inputs or relevant options changed, or when it is listed in `cfg.run.force`.

In [ ]:
import os, warnings
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import heatup
from heatup import cfg, run_step, PhaseDir, override
from heatup.runtime import verbose
from heatup.report import print_layout

warnings.filterwarnings("ignore")
verbose()
print("HeatUp", heatup.__version__)
from heatup.builders import locate_phase, write_phase, fcc
from heatup import database_status
from heatup.md_analysis import analyse_md
from heatup.vdos import run_vdos

## Configuration - the only cell you normally edit\nSections of `cfg`: `calc`, `relax`, `phonon`, `qha`, `fc3`, `elastic`, `md`, `hull`, `run`, `phases` (gates are configured in notebook 02).

In [ ]:
# ── Calculator / model ───────────────────────────────────────────────────────
cfg.calc.backend = os.environ.get("HEATUP_BACKEND", "mace-mp")   # 'mace-mp' | 'chgnet' | 'm3gnet' | 'custom'
cfg.calc.model = os.environ.get("MACE_MODEL_PATH", "medium-mpa-0")  # mace_mp name or path to a .model; legacy 'mace-mpa-0-medium' is normalised
cfg.calc.model_overrides = {}                # per-material fine-tuned model, e.g. {"BaTiO3": "BTO_MACE.model"}; the rest (Ba, Ti, O) use cfg.calc.model
cfg.calc.dispersion = False                  # D3 correction on top of the MLIP
cfg.calc.dtype = "float64"                   # keep float64 for finite-difference phonons / elastic constants
DEVICE = None                                # None -> cuda if usable, else cpu ('cpu' / 'cuda' to force)

# ── Paths ────────────────────────────────────────────────────────────────────
cfg.phases.database_root = "database"
cfg.phases.candidates_root = "input/candidates"   # read-only

# ── Phonon mode (set the same value in notebook 02) ──────────────────────────
#   'HA'   harmonic approximation (default consistent hull)
#   'QHA'  quasi-harmonic (adds the "qha" step: phonons at several volumes)
#   'VDOS' anharmonic VDOS from MD for F_vib of the target
# HA phonons are ALWAYS computed (cheap baseline + competing-phase free energies).
cfg.hull.phonon_method = "HA"
cfg.hull.temperatures = list(range(0, 1501, 50))   # T grid for F(T)

# ── Relaxation ───────────────────────────────────────────────────────────────
cfg.relax.fmax = 0.005              # eV/A
cfg.relax.max_steps = 1000
cfg.relax.relax_cell = True         # relax cell shape and volume too
cfg.relax.constant_volume = False   # keep the volume, relax shape + ions
cfg.relax.fix_symmetry = False       # keep the space group during the relaxation
cfg.relax.symprec = 1e-3
cfg.relax.pressure_GPa = 0.0

# ── Harmonic phonons (phonopy, finite displacements) ─────────────────────────
# Sizing: None = "as needed". An explicit supercell wins (then leave min_length_A None); otherwise the smallest
# supercell with perpendicular widths >= min_length_A is built (None -> built-in 12 A); max_atoms None = no limit.
cfg.phonon.supercell = None         # e.g. (3, 3, 3)
cfg.phonon.min_length_A = None      # e.g. 12.0
cfg.phonon.max_atoms = None         # e.g. 800 as a guard against runaway cells
cfg.phonon.delta_A = 0.01           # displacement amplitude
cfg.phonon.mesh_length_A = 80.0     # q-mesh density for the DOS
cfg.phonon.dos_sigma_meV = 0.5      # plotting smearing only
cfg.phonon.band_npoints = 61
cfg.phonon.symprec = 1e-4
cfg.phonon.imag_tol_meV = 0.5       # imaginary modes shallower than this count as numerical noise

# ── Force-constant order ─────────────────────────────────────────────────────
cfg.phonon.fc_order = 2             # 2 = harmonic only; 3 = also third-order constants (phono3py, very slow): adds the 'fc3' step

# ── Third-order force constants (only used with cfg.phonon.fc_order = 3) ─────
cfg.fc3.supercell = None            # e.g. (2, 2, 2); same rule as above (None -> built-in 8 A)
cfg.fc3.min_length_A = None
cfg.fc3.max_atoms = None            # e.g. 400
cfg.fc3.delta_A = 0.03              # displacement amplitude (larger than for fc2)
cfg.fc3.cutoff_pair_A = None        # ignore atom pairs farther than this (e.g. 4.0): far fewer displaced cells
cfg.fc3.separate_fc2 = True         # fc2 from the harmonic supercell (cfg.phonon), not the small fc3 one
cfg.fc3.kappa_mesh_length_A = 30.0  # q-mesh density for kappa_RTA
cfg.fc3.temperatures = [100.0, 200.0, 300.0, 400.0, 600.0, 800.0, 1000.0]
cfg.fc3.min_freq_THz = 0.5          # modes below this are left out of the gamma/nu statistics
cfg.fc3.ratio_warn = 0.05           # report flag when <gamma/nu> at the operating T exceeds this

# ── QHA (only used with cfg.hull.phonon_method = 'QHA') ──────────────────────
cfg.qha.n_volumes = 7               # odd, >= 5 recommended
cfg.qha.volume_range = 0.06         # +/- fraction of V0
cfg.qha.poly_order = 4              # polynomial order of the F(V) fit at each T
cfg.qha.relax_internal = True       # relax ions at every volume (required)

# ── Elastic tensor ───────────────────────────────────────────────────────────
cfg.elastic.strain = 0.004
cfg.elastic.relax_ions = True       # relaxed-ion constants (clamped ions over-stiffen ferroelectrics)
cfg.elastic.fmax = 0.002

# ── MD ───────────────────────────────────────────────────────────────────────
cfg.md.ensemble = "NPT"             # 'NPT' (cell fluctuates) | 'NVT' (fixed cell)
cfg.md.timestep_fs = 1.0
cfg.md.n_steps = 30_000             # 30 ps at 1 fs; the VDOS wants >= gates.min_production_ps after equilibration
cfg.md.nblock = 5                   # write a frame every 5 steps -> VDOS Nyquist 413 meV (20 would alias O-containing oxides)
cfg.md.equil_ps = 5.0               # discarded before analysis (replaces the old MD_STEP_EQUIV frames)
cfg.md.ttime_fs = 50.0              # thermostat time constant
cfg.md.ptime_fs = 100.0             # barostat time constant (NPT)
cfg.md.pressure_GPa = 0.0
cfg.md.supercell = None             # e.g. (4, 4, 4); explicit wins, min_cell_A then stays None
cfg.md.min_cell_A = 12.0            # minimum supercell edge for MD (the old AIMD_MIN_CELL_ANG); None -> built-in 15 A
cfg.md.max_atoms = None             # e.g. 1500
cfg.md.seed = 12345

# ── Which steps, and force ───────────────────────────────────────────────────
STEPS_TO_RUN = {"relax": True,
                "elastic": True,
                "phonons": True,
                "qha": cfg.hull.phonon_method == "QHA",
                "fc3": cfg.phonon.fc_order == 3,
                "md": True,
                "analysis": True
}
cfg.run.force = []  # recompute whatever the cache says: ["md"], ["relax", "elastic"] or "all"

# ── Run list ─────────────────────────────────────────────────────────────────
USE_QUERY_FILE = False
QUERY_FILE = "next_candidates.json"  # [{"material": ..., "symmetry": ..., "temperature": ...}, ...]
MANUAL_LIST = [                      # one entry per phase; "temperature" may also be a list
    {"material": "Cu", "symmetry": "Fm-3m", "temperature": 300},
    # {"material": "BaTiO3", "symmetry": "P4mm", "temperature": 600},
    # {"material": "Ba", "symmetry": "mp-122", "temperature": 600},
]

## Build the run list\nA POSCAR in `database/<material>/<label>/` is used as is; one found only under `input/candidates/` is copied into the database.

In [ ]:
import json
run_list = json.load(open(QUERY_FILE)) if USE_QUERY_FILE else MANUAL_LIST

phases = []
for entry in run_list:
    mat, label = entry["material"], entry["symmetry"]
    temps = entry["temperature"] if isinstance(entry["temperature"], (list, tuple)) else [entry["temperature"]]
    d = locate_phase(mat, label)
    if d is None:
        print(f"[missing] {mat}/{label}: put a POSCAR in {cfg.phases.database_root}/{mat}/{label}/")
        continue
    phases.append((d, [int(T) for T in temps]))
    print(f"  {d.tag}  T = {temps} K")
print(len(phases), "phase(s) ready")

## Run\nA failed step prints a message and the loop moves on to the next phase.

In [ ]:
print("Calculator :", cfg.calc.backend, "|", cfg.calc.model, "| overrides:", cfg.calc.model_overrides)
print("Steps      :", [s for s, on in STEPS_TO_RUN.items() if on], "| force:", cfg.run.force)

for d, temps in phases:
    print("=" * 70, "\\n", d.tag)
    for step in ("relax", "elastic", "phonons", "qha", "fc3"):
        if STEPS_TO_RUN[step] and not run_step(step, d, device=DEVICE):
            print(f"  [{step}] failed - see the log above")
    if STEPS_TO_RUN["md"]:
        for T in temps:
            if not run_step("md", d, device=DEVICE, temperature=T):
                print(f"  [md {T} K] failed")
    if STEPS_TO_RUN["analysis"]:                                  # block averages + VDOS from the trajectory
        for T in temps:
            analyse_md(d, T)
            run_vdos(d, T)

## What exists now

In [ ]:
display(pd.DataFrame(database_status(cfg.phases.database_root)))
print_layout(cfg.phases.database_root)

### Forcing a rerun\n`cfg.run.force` lists the steps to recompute (`"all"` for every step); `run_step(..., force=True)` does it for one call. Cached steps print `[cached]`.

In [ ]:
with override(**{"run.force": ["relax"]}):                  # recompute only the relaxation, once
    run_step("relax", phases[0][0], device=DEVICE)

## Provenance of the cached results\nEvery step stores the exact options it used.

In [ ]:
from heatup.manifest import manifest_summary
print(manifest_summary(phases[0][0].join("relaxation")))